<div style="background: linear-gradient(135deg, #8338ec 0%, #3a0ca3 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">🔄 02 — Validação Cruzada (K-Fold)</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 02 · Bloco 04 · Ajuste</p>
</div>

## 🎯 Objetivos

- Entender por que 1 split não é suficiente
- Usar K-Fold Cross-Validation corretamente
- Ajustar hiperparâmetros com GridSearchCV

---

## 1️⃣ Por que Cross-Validation?

Um único split pode ser sortudo ou azarado. Cross-validation usa **K splits diferentes**:
```
Fold 1: [TESTE][treino][treino][treino][treino]
Fold 2: [treino][TESTE][treino][treino][treino]
Fold 3: [treino][treino][TESTE][treino][treino]
Fold 4: [treino][treino][treino][TESTE][treino]
Fold 5: [treino][treino][treino][treino][TESTE]

Score final = média dos 5 scores
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

X, y = load_iris(return_X_y=True)

# Comparar modelos com cross-validation
modelos = {
    'Tree (d=3)': DecisionTreeClassifier(max_depth=3),
    'Tree (d=None)': DecisionTreeClassifier(),
    'RF (100)': RandomForestClassifier(n_estimators=100),
}

print(f'{"Modelo":>15} | {"Scores":>30} | {"Média ± Std"}')
print('-' * 70)
for nome, model in modelos.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f'{nome:>15} | {str(scores.round(3)):>30} | {scores.mean():.3f} ± {scores.std():.3f}')

print('\n→ Tree sem limite: alta variância entre folds = instável')

## 2️⃣ GridSearchCV: Busca Automática de Hiperparâmetros

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,  # Usar todos os cores
    verbose=0
)
grid.fit(X, y)

print(f'Melhor score: {grid.best_score_:.4f}')
print(f'Melhores params: {grid.best_params_}')
print(f'Total de combinações testadas: {len(grid.cv_results_["params"])}')

## 3️⃣ Visualizar Resultados do Grid Search

In [ ]:
import pandas as pd

resultados = pd.DataFrame(grid.cv_results_)
pivot = resultados.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_n_estimators',
    aggfunc='max'
)

fig, ax = plt.subplots(figsize=(10, 6))
import seaborn as sns
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax)
ax.set_title('GridSearch: Accuracy por max_depth × n_estimators', fontsize=13, fontweight='bold')
plt.show()

## 🏋️ Questões

### Q1. Use `RandomizedSearchCV` em vez de `GridSearchCV`. Qual é mais rápido? O resultado é similar?

### Q2. Faça GridSearch com `StratifiedKFold(n_splits=10)`. O resultado muda muito com mais folds?

### Q3. Adicione um Pipeline com `StandardScaler` + `SVC` e busque os melhores `C` e `kernel`.

In [ ]:
# Resolva aqui
